# Data Validation

Created using ClaudeAI. Prompt- What all tests should i do to validate the data preprocessing steps? 

In [ ]:
import os
import numpy as np
import pandas as pd

train = pd.read_csv(os.path.join('..', 'data', 'splits', 'train.csv'))
val   = pd.read_csv(os.path.join('..', 'data', 'splits', 'val.csv'))
test  = pd.read_csv(os.path.join('..', 'data', 'splits', 'test.csv'))
all_data = pd.concat([train, val, test], ignore_index=True)
processed = pd.read_csv(os.path.join('..', 'data', 'features', 'features_processed.csv'))
cohort = pd.read_csv(os.path.join('..', 'data', 'cohort', 'sepsis3_cohort.csv'))

print(f'Train: {train.admissionid.nunique():,} admissions, {len(train):,} rows')
print(f'Val:   {val.admissionid.nunique():,} admissions, {len(val):,} rows')
print(f'Test:  {test.admissionid.nunique():,} admissions, {len(test):,} rows')

## Sepsis-3 criteria



In [ ]:
print('=== Sepsis-3 Criteria ===')

# SOFA >= 2
violations = cohort[cohort['sofa_total_score'] < 2]
status = 'PASS' if len(violations) == 0 else f'FAIL ({len(violations)} patients)'
print(f'SOFA >= 2 for all patients: {status}')
print(cohort['sofa_total_score'].describe().round(1))

# Infection criteria
cohort['any_infection'] = (
    (cohort['sepsis_at_admission'] == 1) |
    (cohort['sepsis_antibiotics_bool'] == 1) |
    ((cohort['other_antibiotics_bool'] == 1) & (cohort['sepsis_cultures_bool'] == 1)) |
    (cohort['diagnosis'].str.contains(
        'sepsis|pneumon|infecti|peritonitis|meningitis|endocard|cholangitis|abscess',
        case=False, na=False
    ))
)
violations = cohort[~cohort['any_infection']]
status = 'PASS' if len(violations) == 0 else f'FAIL ({len(violations)} patients)'
print(f'\nAll patients meet infection criterion: {status}')

print('\nSOFA component scores:')
sofa_cols = [c for c in ['sofa_respiration_score','sofa_coagulation_score','sofa_liver_score',
                          'sofa_cardiovascular_score','sofa_cns_score','sofa_renal_score'] if c in cohort.columns]
print(cohort[sofa_cols].describe().round(1))

## Data integrity checks

In [ ]:
print('=== No patient leakage across splits ===')
train_ids = set(train['admissionid'])
val_ids   = set(val['admissionid'])
test_ids  = set(test['admissionid'])

for a, b, name in [(train_ids, val_ids, 'train/val'), (train_ids, test_ids, 'train/test'), (val_ids, test_ids, 'val/test')]:
    overlap = a & b
    print(f'{name}: {"PASS" if not overlap else f"FAIL ({len(overlap)} shared)"}')

In [ ]:
print('=== No missing values in clinical features ===')
meta = ['admissionid','window','start_time','90day_mortality','LOS_remaining_days',
        'rrt_next_timestep','mv_next_timestep','admission_age_norm','weight_norm',
        'weight_null','height_norm','height_null','gender']
feat_cols = [c for c in train.columns if c not in meta]

missing = all_data[feat_cols].isna().sum()
missing = missing[missing > 0]
if len(missing) == 0:
    print(f'PASS: No missing values in {len(feat_cols)} feature columns')
else:
    print(f'FAIL:')
    print(missing.to_string())

In [ ]:
print('=== Binary labels are 0/1 ===')
for label in ['90day_mortality', 'rrt_next_timestep', 'mv_next_timestep']:
    if label not in all_data.columns:
        continue
    vals = set(all_data[label].unique())
    print(f'{label}: {"PASS" if vals <= {0,1} else f"FAIL {vals}"}')

In [ ]:
print('=== Sequential windows per patient ===')
bad = []
for aid, grp in all_data.groupby('admissionid'):
    w = sorted(grp['window'].tolist())
    if w != list(range(len(w))):
        bad.append(aid)
print(f'{"PASS" if not bad else f"FAIL ({len(bad)} patients)"}: all windows sequential from 0')

In [ ]:
print('=== Clinical value ranges ===')
checks = {
    'heart_rate': (30, 180), 'sysbp': (40, 300), 'meanbp': (20, 160),
    'tempc': (33, 42), 'spo2': (74, 100), 'ph': (7.0, 8.5), 'lactate': (0, 30),
}
for col, (lo, hi) in checks.items():
    if col not in all_data.columns: continue
    out = ((all_data[col] < lo) | (all_data[col] > hi)).sum()
    print(f'{col}: {"PASS" if out == 0 else f"FAIL ({out} out of range)"}')

In [ ]:
print('=== Split proportions ===')
total = len(all_data)
for name, d in [('Train', train), ('Val', val), ('Test', test)]:
    pct = len(d)/total
    print(f'{name}: {pct:.1%}')

rates = [train['90day_mortality'].mean(), val['90day_mortality'].mean(), test['90day_mortality'].mean()]
spread = max(rates) - min(rates)
print(f'\nMortality rate spread across splits: {spread:.3f} ({"PASS" if spread < 0.05 else "WARN"})')

In [ ]:
print('=== Static features consistent within patient ===')
for col in ['gender', 'admission_age_norm']:
    if col not in all_data.columns: continue
    inconsistent = all_data.groupby('admissionid')[col].nunique()
    bad = inconsistent[inconsistent > 1]
    print(f'{col}: {"PASS" if len(bad)==0 else f"FAIL ({len(bad)} patients)"}')

In [ ]:
print('=== Row count integrity ===')
total_splits = len(train) + len(val) + len(test)
print(f'Splits total: {total_splits:,}  Processed CSV: {len(processed):,}')
print(f'{"PASS" if total_splits == len(processed) else "FAIL: mismatch"}')

## Summary

In [ ]:
print('=== Summary ===')
print(f'Admissions: {all_data.admissionid.nunique():,}')
print(f'Windows:    {len(all_data):,}')
print(f'Features:   {len(feat_cols)}')
print(f'90-day mortality:  {all_data["90day_mortality"].mean():.3f}')
print(f'Next-step RRT:     {all_data["rrt_next_timestep"].mean():.3f}')
print(f'Next-step MV:      {all_data["mv_next_timestep"].mean():.3f}')
print(f'LOS median:        {all_data["LOS_remaining_days"].median():.1f} days')
print(f'Split: {train.admissionid.nunique()} / {val.admissionid.nunique()} / {test.admissionid.nunique()} (train/val/test)')